In [1]:
import os
from pathlib import Path
import numpy as np
import scipy.io
import matplotlib.pyplot as plt

In [2]:
# set Human SAO directory here
SAO_BASE_DIR_NAME = 'F:/Ionosonde/SAO_pick'
SAO_BASE_DIR = Path(SAO_BASE_DIR_NAME)

# set SBF directory here
SBF_BASE_DIR_NAME = 'F:/Ionosonde/SBF_mat'
SBF_BASE_DIR = Path(SBF_BASE_DIR_NAME)

# set ARTIST SAO directory here
SAO_AUTO_DIR_NAME = 'F:/Ionosonde/SAO_pick_ARTIST/'
SAO_AUTO_DIR = Path(SAO_AUTO_DIR_NAME)

In [3]:
counter = 0
# save list
gt_x_list = list()
gt_y_list = list()
auto_y_list = list()
for SAOs in SAO_BASE_DIR.glob('*.mat'):
    SAO_day = scipy.io.loadmat(str(SAOs))
    F2O = SAO_day['t_F2O']
    F1O = SAO_day['t_F1O']
    EO = SAO_day['t_EO']

    if os.path.exists(SAO_AUTO_DIR_NAME+SAOs.name[:-11]+'.SAO.mat'):
        pass
    else:
        continue
    # load SAO mat
    SAO_day_auto = scipy.io.loadmat(SAO_AUTO_DIR_NAME+SAOs.name[:-11]+'.SAO.mat')
    F2O_auto = SAO_day_auto['t_F2O']
    F1O_auto = SAO_day_auto['t_F1O']
    EO_auto = SAO_day_auto['t_EO']

    if len(F1O) < 1:
        continue
    for idx in range(len(SAO_day['TS'])-1):
        t_TS = SAO_day['TS'][idx+1]
        timecode = SAO_day['TS'][idx+1][2:6]+SAO_day['TS'][idx+1][9:19]
        t_mat_name = SBF_BASE_DIR_NAME + '/WU430_' + timecode + '.SBF.mat'
        
        if os.path.exists(t_mat_name):
            pass
        else:
            continue
        
        t_mat = scipy.io.loadmat(t_mat_name)

        amp_O = t_mat['AmplitudeO']
        amp_X = t_mat['AmplitudeX']
        MPA_O = t_mat['MPAO']
        MPA_X = t_mat['MPAX']
        Hs = float(t_mat['HeightStart'][0][0])
        Hd = float(t_mat['HeightStep'][0][0])
        Fs = float(t_mat['FrequencyStart'][0][0])
        Fd = float(t_mat['FrequencyStep'][0][0])

        input_array = np.zeros([np.shape(amp_O)[0],np.shape(amp_O)[1],3],dtype='uint8')
        input_array[:,:,0] = amp_O
        input_array[:,:,1] = amp_X

        if np.shape(input_array)[1] != np.shape(MPA_O)[1]:
            for jdx in range(np.shape(amp_O)[0]):
                input_array[jdx,:,2] = MPA_O[0,:np.shape(input_array)[1]]
        else:
            for jdx in range(np.shape(amp_O)[0]):
                input_array[jdx,:,2] = MPA_O

        print(timecode)

        t_mask = np.zeros([np.shape(amp_O)[0],np.shape(amp_O)[1],3],dtype='uint8')

        EO_length = int(len(EO[idx*2+1])/8)

        EO_freq = []
        EO_height = []
        for kdx in range(EO_length):
            #print(F2O[idx*2+1][kdx*8:(kdx+1)*8])
            try:
                if float(EO[idx*2+2][kdx*8:(kdx+1)*8]) > 3000:
                    continue
                EO_freq.append((float(EO[idx*2+1][kdx*8:(kdx+1)*8])-Fs)/Fd)
                EO_height.append((float(EO[idx*2+2][kdx*8:(kdx+1)*8])-Hs)/Hd)
            except:
                break
        EO_length = len(EO_freq)
        F1O_length = int(len(F1O[idx*2+1])/8)

        F1O_freq = []
        F1O_height = []
        for kdx in range(F1O_length):
            #print(F2O[idx*2+1][kdx*8:(kdx+1)*8])
            try:
                if float(F1O[idx*2+2][kdx*8:(kdx+1)*8]) > 3000:
                    continue
                F1O_freq.append((float(F1O[idx*2+1][kdx*8:(kdx+1)*8])-Fs)/Fd)
                F1O_height.append((float(F1O[idx*2+2][kdx*8:(kdx+1)*8])-Hs)/Hd)
            except:
                break
        F1O_length = len(F1O_freq)
        F2O_length = int(len(F2O[idx*2+1])/8)
        
        F2O_freq = []
        F2O_height = []
        for kdx in range(F2O_length):
            #print(F2O[idx*2+1][kdx*8:(kdx+1)*8])
            try:
                if float(F2O[idx*2+2][kdx*8:(kdx+1)*8]) > 3000:
                    continue
                F2O_freq.append((float(F2O[idx*2+1][kdx*8:(kdx+1)*8])-Fs)/Fd)
                F2O_height.append((float(F2O[idx*2+2][kdx*8:(kdx+1)*8])-Hs)/Hd)
            except:
                break
        F2O_length = len(F2O_freq)
        
        for kdx in range(len(EO_freq)-1):
            t_gap = int(EO_freq[kdx+1]) - int(EO_freq[kdx])
            t_k = int(EO_height[kdx+1]) - int(EO_height[kdx])
            t_mask[int(EO_height[kdx]),int(EO_freq[kdx]),0] = 1
            for mdx in range(t_gap):
                t_mask[int(EO_height[kdx]+(mdx+1)*t_k),int(EO_freq[kdx]+mdx+1),0] = 1

        for kdx in range(len(F1O_freq)-1):
            t_gap = int(F1O_freq[kdx+1]) - int(F1O_freq[kdx])
            t_k = int(F1O_height[kdx+1]) - int(F1O_height[kdx])
            t_mask[int(F1O_height[kdx]),int(F1O_freq[kdx]),1] = 1
            for mdx in range(t_gap):
                t_mask[int(F1O_height[kdx])+(mdx+1)*t_k,int(F1O_freq[kdx]+mdx+1),1] = 1


        for kdx in range(len(F2O_freq)-1):
            t_gap = int(F2O_freq[kdx+1]) - int(F2O_freq[kdx])
            t_k = int(F2O_height[kdx+1]) - int(F2O_height[kdx])
            t_mask[int(F2O_height[kdx]),int(F2O_freq[kdx]),2] = 1
            for mdx in range(t_gap):
                t_mask[int(F2O_height[kdx])+(mdx+1)*t_k,int(F2O_freq[kdx]+mdx+1),2] = 1

        #auto mask
        auto_mask = np.zeros([np.shape(amp_O)[0],np.shape(amp_O)[1],3],dtype='uint8')
        try:
            EO_length_auto = int(len(EO_auto[idx*2+1])/8)
        except:
            EO_length_auto = 0
        EO_freq = []
        EO_height = []
        for kdx in range(EO_length_auto):
            #print(F2O[idx*2+1][kdx*8:(kdx+1)*8])
            try:
                if float(EO_auto[idx*2+2][kdx*8:(kdx+1)*8]) > 3000:
                    continue
                EO_freq.append((float(EO_auto[idx*2+1][kdx*8:(kdx+1)*8])-Fs)/Fd)
                EO_height.append((float(EO_auto[idx*2+2][kdx*8:(kdx+1)*8])-Hs)/Hd)
            except:
                break
        EO_length_auto = len(EO_freq)

        try:
            F1O_length_auto = int(len(F1O_auto[idx*2+1])/8)
        except:
            F1O_length_auto = 0
            
        F1O_freq = []
        F1O_height = []
        for kdx in range(F1O_length_auto):
            #print(F2O[idx*2+1][kdx*8:(kdx+1)*8])
            try:
                if float(F1O_auto[idx*2+2][kdx*8:(kdx+1)*8]) > 3000:
                    continue
                F1O_freq.append((float(F1O_auto[idx*2+1][kdx*8:(kdx+1)*8])-Fs)/Fd)
                F1O_height.append((float(F1O_auto[idx*2+2][kdx*8:(kdx+1)*8])-Hs)/Hd)
            except:
                break
        F1O_length_auto = len(F1O_freq)

        try:
            F2O_length_auto = int(len(F2O_auto[idx*2+1])/8)
        except:
            F2O_length_auto = 0
        
        F2O_freq = []
        F2O_height = []
        for kdx in range(F2O_length_auto):
            #print(F2O[idx*2+1][kdx*8:(kdx+1)*8])
            try:
                if float(F2O_auto[idx*2+2][kdx*8:(kdx+1)*8]) > 3000:
                    continue
                F2O_freq.append((float(F2O_auto[idx*2+1][kdx*8:(kdx+1)*8])-Fs)/Fd)
                F2O_height.append((float(F2O_auto[idx*2+2][kdx*8:(kdx+1)*8])-Hs)/Hd)
            except:
                break
        F2O_length_auto = len(F2O_freq)
        
        for kdx in range(len(EO_freq)):
            auto_mask[int(EO_height[kdx]),int(EO_freq[kdx]),0] = 1.0
           
        for kdx in range(len(F1O_freq)):
            auto_mask[int(F1O_height[kdx]),int(F1O_freq[kdx]),1] = 1.0

        for kdx in range(len(F2O_freq)):
            auto_mask[int(F2O_height[kdx]),int(F2O_freq[kdx]),2] = 1.0       

        gt_x_list.append(input_array)
        gt_y_list.append(t_mask)
        auto_y_list.append(auto_mask)

In [6]:
c_id = 7000
plt.figure(figsize=(12,12))
plt.imshow(auto_y_list[-1]*255)
plt.show()
plt.figure(figsize=(12,12))
plt.imshow(gt_y_list[-1]*255)
plt.show()

IndexError: list index out of range

In [24]:
len(gt_x_list)

19910

In [7]:
check_id = np.random.randint(low=0,high=len(gt_x_list))
plt.figure(figsize=(12,12))
plt.subplot(1,2,1)
plt.imshow(gt_x_list[check_id][:,:,1],cmap='gray')
plt.gca().invert_yaxis()
plt.subplot(1,2,2)
plt.imshow(gt_y_list[check_id][:,:,2],cmap='gray')
plt.gca().invert_yaxis()
plt.show()
plt.close()

ValueError: low >= high

In [4]:
import pickle

In [ ]:
pickle.dump(gt_x_list,open('gt_x_list_uint8_1229.pickle','wb'))

In [ ]:
pickle.dump(gt_y_list,open('gt_y_list_uint8_1229.pickle','wb'))

In [ ]:
pickle.dump(auto_y_list,open('auto_y_list_uint8_1229.pickle','wb'))